# LeetCode #80 — Remove Duplicates from Sorted Array II: giữ tối đa 2 bản, hai con trỏ

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers

Skeleton trên LeetCode:

```python
class Solution:
    def removeDuplicates(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Mẫu **hai con trỏ đọc / ghi** (read / write) để lọc mảng tại chỗ.
2. Điều kiện gọn: so phần tử đang đọc với **`nums[write - 2]`**.
3. Tổng quát hóa: giữ tối đa `k` bản (bài #26 là `k = 1`).
4. Hiểu "trả về `k`, LeetCode chỉ kiểm tra `k` phần tử đầu".

# 1. Bài toán là gì?

Cho mảng `nums` đã sắp xếp **không giảm**. Xóa **tại chỗ** các phần tử trùng sao cho mỗi giá trị xuất hiện **tối đa hai lần**. Thứ tự tương đối giữ nguyên.

Trả về `k` — số phần tử còn lại. `k` phần tử **đầu** của `nums` phải là kết quả; phần sau không quan trọng. Bộ nhớ phụ `O(1)`.

**Constraints (theo đề):**

- `1 <= nums.length <= 3 * 10^4`
- `-10^4 <= nums[i] <= 10^4`
- `nums` được sắp xếp không giảm.

---

## Ví dụ

```text
nums = [1, 1, 1, 2, 2, 3]           -> k = 5, nums = [1, 1, 2, 2, 3, _]
nums = [0, 0, 1, 1, 1, 1, 2, 3, 3]  -> k = 7, nums = [0, 0, 1, 1, 2, 3, 3, _, _]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Hai con trỏ (two pointers)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [1]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [2]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.2 Áp dụng vào bài này

**Con trỏ đọc/ghi** trên mảng đã sắp xếp: phần tử `x` được ghi khi `write < 2` hoặc `x != nums[write − 2]` — tức là chưa có đủ 2 bản sao trong phần đã ghi.

# 3. Cách LeetCode chấm

Bộ chấm (custom judge) làm như sau:

```python
k = solution.removeDuplicates(nums)
assert k == len(expected)
for i in range(k):
    assert nums[i] == expected[i]
```

Nên ta phải **vừa** trả về `k` đúng, **vừa** sửa `nums[:k]` đúng. Ta mô phỏng bộ chấm bằng hàm `judge` ở mục 7.

# 4. Baseline: đếm và dựng lại (dùng thêm bộ nhớ)

Dễ hiểu nhưng tốn `O(n)` bộ nhớ phụ — vi phạm yêu cầu.

In [3]:
from typing import List
import random

def remove_dup_extra(nums: List[int]) -> int:
    result = []
    for x in nums:
        if len(result) < 2 or not (result[-1] == x and result[-2] == x):
            result.append(x)
    nums[:len(result)] = result
    return len(result)

a = [1, 1, 1, 2, 2, 3]
k = remove_dup_extra(a)
print(k, a[:k])

5 [1, 1, 2, 2, 3]


# 5. Ý tưởng chính: hai con trỏ `write` và `read`

- `read` duyệt từng phần tử.
- `write` là vị trí ghi tiếp theo; `nums[:write]` là kết quả đã lọc.

Khi nào giữ `nums[read]`? Khi kết quả hiện tại **chưa** có đủ hai bản của giá trị này. Vì mảng đã sắp xếp, các bản giống nhau đứng liền nhau trong kết quả, nên chỉ cần nhìn **hai phần tử cuối** của kết quả. Mà nếu `nums[write - 2] == x` thì (do sắp xếp) `nums[write - 1] == x` luôn. Điều kiện chỉ còn:

```text
giữ x  ⇔  write < 2  hoặc  nums[write - 2] != x
```

In [4]:
def remove_dup_two_pointers(nums: List[int]) -> int:
    write = 0
    for x in nums:
        if write < 2 or nums[write - 2] != x:
            nums[write] = x
            write += 1
    return write

a = [0, 0, 1, 1, 1, 1, 2, 3, 3]
k = remove_dup_two_pointers(a)
print(k, a[:k])

7 [0, 0, 1, 1, 2, 3, 3]


`for x in nums` vẫn an toàn dù ta ghi vào `nums`: `write <= read` luôn đúng, nên ta chỉ ghi vào vị trí **đã đọc qua**.

# 6. Mô phỏng

In [5]:
def trace_remove(nums: List[int]) -> int:
    write = 0
    print(f"{'read':>4} {'x':>3} {'nums[write-2]':>14}  hành động        nums[:write]")
    for read, x in enumerate(nums):
        ref = nums[write - 2] if write >= 2 else "-"
        if write < 2 or nums[write - 2] != x:
            nums[write] = x
            write += 1
            act = "giữ"
        else:
            act = "bỏ (đã đủ 2)"
        print(f"{read:>4} {x:>3} {str(ref):>14}  {act:<15}  {nums[:write]}")
    print("k =", write)
    return write

_ = trace_remove([1, 1, 1, 2, 2, 3])

read   x  nums[write-2]  hành động        nums[:write]
   0   1              -  giữ              [1]
   1   1              -  giữ              [1, 1]
   2   1              1  bỏ (đã đủ 2)     [1, 1]
   3   2              1  giữ              [1, 1, 2]
   4   2              1  giữ              [1, 1, 2, 2]
   5   3              2  giữ              [1, 1, 2, 2, 3]
k = 5


Chú ý dòng `read = 2`: `x = 1`, `nums[write - 2] = nums[0] = 1` → bằng nhau → đã có hai số 1 → bỏ.

# 7. Phiên bản đúng format LeetCode

In [6]:
class Solution:
    def removeDuplicates(self, nums: List[int]) -> int:
        write = 0
        for x in nums:
            if write < 2 or nums[write - 2] != x:
                nums[write] = x
                write += 1
        return write

In [7]:
def expected_of(nums, k_max=2):
    res = []
    for x in nums:
        if res.count(x) < k_max:
            res.append(x)
    return res

def judge(f, nums):
    nums = nums[:]
    expected = expected_of(nums)
    k = f(nums)
    assert k == len(expected), (k, expected)
    assert nums[:k] == expected, (nums[:k], expected)
    return k, nums[:k]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", judge(sol.removeDuplicates, [1, 1, 1, 2, 2, 3]), (5, [1, 1, 2, 2, 3]))
check("Ví dụ 2", judge(sol.removeDuplicates, [0, 0, 1, 1, 1, 1, 2, 3, 3]), (7, [0, 0, 1, 1, 2, 3, 3]))
check("Một phần tử", judge(sol.removeDuplicates, [5]), (1, [5]))
check("Toàn giống nhau", judge(sol.removeDuplicates, [7] * 6), (2, [7, 7]))
check("Không trùng", judge(sol.removeDuplicates, [1, 2, 3]), (3, [1, 2, 3]))
check("Số âm", judge(sol.removeDuplicates, [-3, -3, -3, -1]), (3, [-3, -3, -1]))

random.seed(80)
for _ in range(5000):
    nums = sorted(random.randint(-3, 3) for _ in range(random.randint(1, 15)))
    judge(sol.removeDuplicates, nums)
    judge(remove_dup_extra, nums)
print("5000 mảng ngẫu nhiên: PASS bộ chấm mô phỏng")

Ví dụ 1              PASS  (5, [1, 1, 2, 2, 3])
Ví dụ 2              PASS  (7, [0, 0, 1, 1, 2, 3, 3])
Một phần tử          PASS  (1, [5])
Toàn giống nhau      PASS  (2, [7, 7])
Không trùng          PASS  (3, [1, 2, 3])
Số âm                PASS  (3, [-3, -3, -1])
5000 mảng ngẫu nhiên: PASS bộ chấm mô phỏng


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** sau khi xử lý `nums[0..read]`, `nums[:write]` bằng kết quả đúng cho tiền tố đó.

- `nums[:write]` sắp xếp không giảm và mỗi giá trị xuất hiện ≤ 2 lần.
- Phần tử mới `x ≥` mọi phần tử trong `nums[:write]`. Số bản `x` đã có bằng 2 ⇔ hai phần tử cuối là `x` ⇔ `nums[write - 2] == x` (vì nếu phần tử thứ hai từ cuối là `x` thì phần tử cuối, nằm giữa nó và `x`, cũng là `x`).

```text
Time:  O(n)  — một lượt
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: So với `nums[read - 2]` (mảng gốc) thay vì `nums[write - 2]` (mảng kết quả)

In [8]:
def wrong_compare_read(nums: List[int]) -> int:
    write = 0
    for read in range(len(nums)):
        if read < 2 or nums[read] != nums[read - 2]:   # SAI: nums[read-2] có thể đã bị ghi đè
            nums[write] = nums[read]
            write += 1
    return write

for t in ([1, 1, 1, 2, 2, 3], [1, 1, 1, 2, 2, 2, 3, 3]):
    a = t[:]
    k = wrong_compare_read(a)
    print("Sai :", k, a[:k], "| Đúng:", judge(sol.removeDuplicates, t))

Sai : 4 [1, 1, 2, 3] | Đúng: (5, [1, 1, 2, 2, 3])
Sai : 5 [1, 1, 2, 3, 3] | Đúng: (6, [1, 1, 2, 2, 3, 3])


Ngay ví dụ 1 đã sai. Ở `read = 3`, số `2` được ghi vào `nums[2]` (đè lên số `1` gốc). Đến `read = 4`, bản sai so `nums[4] = 2` với `nums[2]` — giờ là `2` (đã bị ghi đè), không còn là `1` gốc — nên bỏ nhầm số `2` thứ hai.

## Lỗi 2: Chỉ so với phần tử liền trước (giữ 1 bản — đó là bài #26)

In [9]:
def wrong_prev_only(nums: List[int]) -> int:
    write = 0
    for x in nums:
        if write < 1 or nums[write - 1] != x:       # SAI: đây là #26
            nums[write] = x
            write += 1
    return write

a = [1, 1, 1, 2, 2, 3]
k = wrong_prev_only(a)
print("Sai :", k, a[:k])

Sai : 3 [1, 2, 3]


## Lỗi 3: Dùng `del nums[i]` / `nums.pop(i)` trong vòng lặp

Mỗi lần xóa tốn `O(n)` (dời các phần tử sau) → tổng `O(n²)`; còn dễ sai chỉ số vì mảng co lại trong lúc duyệt.

## Lỗi 4: Trả về `nums` thay vì `k`

Đề yêu cầu trả **số nguyên** `k`.

# 10. Mẹo nhận diện pattern

**Hai con trỏ đọc / ghi** khi cần **lọc** mảng tại chỗ, giữ thứ tự:

```python
write = 0
for x in nums:
    if giữ(x, nums[:write]):
        nums[write] = x
        write += 1
return write
```

Tổng quát "giữ tối đa `k` bản" trên mảng đã sắp xếp: `if write < k or nums[write - k] != x`.

Bài liên quan: **#26 Remove Duplicates from Sorted Array** (`k = 1`), **#27 Remove Element**, **#283 Move Zeroes**, **#1047 Remove All Adjacent Duplicates In String** (dùng stack).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy thuật toán trên `[1, 1, 1, 1, 2, 3, 3, 3]`. Kết quả `k` và `nums[:k]`?

## Bài 2 — Điền chỗ trống

In [10]:
def exercise_2_remove_duplicates(nums: List[int]) -> int:
    write = 0
    for x in nums:
        # TODO: điều kiện giữ x, ghi và tăng write
        pass
    return write


# a = [1, 1, 1, 2, 2, 3]; k = exercise_2_remove_duplicates(a); assert (k, a[:k]) == (5, [1, 1, 2, 2, 3])
# a = [0, 0, 1, 1, 1, 1, 2, 3, 3]; k = exercise_2_remove_duplicates(a); assert (k, a[:k]) == (7, [0, 0, 1, 1, 2, 3, 3])

## Bài 3 — Tổng quát: giữ tối đa `k` bản

In [11]:
def exercise_3_remove_keep_k(nums: List[int], k: int) -> int:
    # TODO
    pass


# a = [1, 1, 1, 2, 2, 3]; r = exercise_3_remove_keep_k(a, 1); assert (r, a[:r]) == (3, [1, 2, 3])
# a = [1, 1, 1, 1, 2]; r = exercise_3_remove_keep_k(a, 3); assert (r, a[:r]) == (4, [1, 1, 1, 2])
# a = [4, 4]; r = exercise_3_remove_keep_k(a, 5); assert (r, a[:r]) == (2, [4, 4])

## Bài 4 — Biến thể: mảng **chưa** sắp xếp

Mảng **không** sắp xếp; giữ tối đa 2 lần xuất hiện **đầu tiên** của mỗi giá trị, giữ thứ tự. Tại chỗ, được dùng `dict` đếm (`O(n)` bộ nhớ).

```python
a = [3, 1, 3, 2, 3, 1, 1]  -> k = 5, a[:5] = [3, 1, 3, 2, 1]
```

In [12]:
def exercise_4_remove_unsorted(nums: List[int]) -> int:
    # TODO
    pass


# a = [3, 1, 3, 2, 3, 1, 1]; k = exercise_4_remove_unsorted(a); assert (k, a[:k]) == (5, [3, 1, 3, 2, 1])
# a = [1]; k = exercise_4_remove_unsorted(a); assert (k, a[:k]) == (1, [1])

## Bài 5 — Mini challenge: Remove All Adjacent Duplicates in String II (LeetCode #1209)

Lặp lại việc xóa `k` ký tự **giống nhau liền kề** cho tới khi không còn xóa được.

```python
remove_k_adjacent("abcd", 2)           -> "abcd"
remove_k_adjacent("deeedbbcccbdaa", 3) -> "aa"
remove_k_adjacent("pbbcggttciiippooaais", 2) -> "ps"
```

Gợi ý nhanh: stack các cặp `[ký tự, số lần liên tiếp]`.

In [13]:
def exercise_5_remove_k_adjacent(s: str, k: int) -> str:
    # TODO
    pass


# assert exercise_5_remove_k_adjacent("abcd", 2) == "abcd"
# assert exercise_5_remove_k_adjacent("deeedbbcccbdaa", 3) == "aa"
# assert exercise_5_remove_k_adjacent("pbbcggttciiippooaais", 2) == "ps"

# 12. Gợi ý

**Bài 1**
- Mức 1: dùng `trace_remove` sau khi tự làm.

**Bài 3**
- Mức 1: thay số 2 bằng `k`: `if write < k or nums[write - k] != x`.

**Bài 4**
- Mức 1: `count = {}`; với mỗi `x`, nếu `count.get(x, 0) < 2` thì ghi và tăng đếm.

**Bài 5**
- Mức 1: duyệt ký tự `ch`: nếu đỉnh stack có cùng ký tự thì tăng đếm, ngược lại push `[ch, 1]`.
- Mức 2: nếu đếm đỉnh bằng `k` thì pop.
- Mức 3: kết quả = nối `ch * cnt` của các phần tử stack.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [14]:
_ = trace_remove([1, 1, 1, 1, 2, 3, 3, 3])

read   x  nums[write-2]  hành động        nums[:write]
   0   1              -  giữ              [1]
   1   1              -  giữ              [1, 1]
   2   1              1  bỏ (đã đủ 2)     [1, 1]
   3   1              1  bỏ (đã đủ 2)     [1, 1]
   4   2              1  giữ              [1, 1, 2]
   5   3              1  giữ              [1, 1, 2, 3]
   6   3              2  giữ              [1, 1, 2, 3, 3]
   7   3              3  bỏ (đã đủ 2)     [1, 1, 2, 3, 3]
k = 5


## Đáp án Bài 2

In [15]:
def answer_2_remove_duplicates(nums: List[int]) -> int:
    return remove_dup_two_pointers(nums)

a = [1, 1, 1, 2, 2, 3]; k = answer_2_remove_duplicates(a); assert (k, a[:k]) == (5, [1, 1, 2, 2, 3])
a = [0, 0, 1, 1, 1, 1, 2, 3, 3]; k = answer_2_remove_duplicates(a); assert (k, a[:k]) == (7, [0, 0, 1, 1, 2, 3, 3])
print("Bài 2 PASS (giống remove_dup_two_pointers ở mục 5)")

Bài 2 PASS (giống remove_dup_two_pointers ở mục 5)


## Đáp án Bài 3

In [16]:
def answer_3_remove_keep_k(nums: List[int], k: int) -> int:
    write = 0
    for x in nums:
        if write < k or nums[write - k] != x:
            nums[write] = x
            write += 1
    return write

a = [1, 1, 1, 2, 2, 3]; r = answer_3_remove_keep_k(a, 1); assert (r, a[:r]) == (3, [1, 2, 3])
a = [1, 1, 1, 1, 2]; r = answer_3_remove_keep_k(a, 3); assert (r, a[:r]) == (4, [1, 1, 1, 2])
a = [4, 4]; r = answer_3_remove_keep_k(a, 5); assert (r, a[:r]) == (2, [4, 4])
for _ in range(3000):
    k = random.randint(1, 4)
    nums = sorted(random.randint(0, 3) for _ in range(random.randint(1, 12)))
    a = nums[:]
    r = answer_3_remove_keep_k(a, k)
    assert a[:r] == expected_of(nums, k)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [17]:
def answer_4_remove_unsorted(nums: List[int]) -> int:
    count = {}
    write = 0
    for x in nums:
        if count.get(x, 0) < 2:
            count[x] = count.get(x, 0) + 1
            nums[write] = x
            write += 1
    return write

a = [3, 1, 3, 2, 3, 1, 1]; k = answer_4_remove_unsorted(a); assert (k, a[:k]) == (5, [3, 1, 3, 2, 1])
a = [1]; k = answer_4_remove_unsorted(a); assert (k, a[:k]) == (1, [1])
print("Bài 4 PASS")

Bài 4 PASS


Khi mảng **không** sắp xếp, mẹo `nums[write - 2]` không còn đúng (các bản giống nhau không liền nhau) → cần bộ đếm.

## Đáp án Bài 5

In [18]:
def answer_5_remove_k_adjacent(s: str, k: int) -> str:
    stack = []
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1
            if stack[-1][1] == k:
                stack.pop()
        else:
            stack.append([ch, 1])
    return "".join(ch * cnt for ch, cnt in stack)

assert answer_5_remove_k_adjacent("abcd", 2) == "abcd"
assert answer_5_remove_k_adjacent("deeedbbcccbdaa", 3) == "aa"
assert answer_5_remove_k_adjacent("pbbcggttciiippooaais", 2) == "ps"
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Hai con trỏ đọc/ghi. Giữ `x` khi `write < 2` hoặc `nums[write - 2] != x` — so với **mảng kết quả**, không phải mảng gốc. Tổng quát: thay 2 bằng `k`.

---

## Checklist trước khi nộp

- [ ] So với `nums[write - 2]`, không phải `nums[read - 2]`?
- [ ] Trả về `k` (số nguyên), và `nums[:k]` đúng?
- [ ] Không dùng `del` / `pop` trong vòng lặp?
- [ ] `O(1)` bộ nhớ phụ?